# Handwriting BCI — Dataset Explorer

Loads the preprocessed single-character trials and the train/val/test splits,
then walks through key data properties, class distributions, example neural
trajectories, and feature representations.

**Run from the project root** (`handwriting_3questions/`) — the cell below
handles `chdir` automatically.

In [ ]:
import os
import sys
from pathlib import Path

# ── Make sure we are at the project root regardless of how the notebook was launched
PROJ_ROOT = Path("__file__").resolve().parent  # works in .py
# For Jupyter, __file__ is not defined — use the notebook's own path instead
try:
    PROJ_ROOT = Path(__file__).resolve().parent
except NameError:
    # We are inside Jupyter; __file__ does not exist.
    # Try to detect from the notebook's own working dir.
    PROJ_ROOT = Path.cwd()
    # Walk up until we find configs/config.yaml
    for p in [PROJ_ROOT] + list(PROJ_ROOT.parents):
        if (p / "configs" / "config.yaml").exists():
            PROJ_ROOT = p
            break

os.chdir(PROJ_ROOT)
sys.path.insert(0, str(PROJ_ROOT / "src"))
print(f"Project root : {PROJ_ROOT}")
print(f"Working dir  : {Path.cwd()}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

from utils_data import load_config, load_processed, load_splits

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1  Config

In [ ]:
cfg = load_config()

print(f"Dataset root   : {cfg['dataset_root']}")
print(f"Sessions       : {len(cfg['sessions'])}")
print(f"Characters     : {len(cfg['characters'])}")
print(f"T_fixed        : {cfg['T_fixed']} bins  ({cfg['T_fixed'] * cfg['dt_ms']:.0f} ms)")
print(f"n_channels     : {cfg['n_channels']}")
print(f"Smoothing σ    : {cfg['sigma_ms']} ms  → {cfg['sigma_ms']/cfg['dt_ms']:.1f} bins")
print(f"Movement window: [{cfg['move_win_start']}, {cfg['move_win_end']})  "
      f"(go-cue at bin {cfg['go_bin']})")

## 2  Load preprocessed trials

In [ ]:
trials, labels, session_ids, block_ids, trial_ids = load_processed(cfg)

N, T, C = trials.shape
print(f"trials shape  : {trials.shape}  (N x T x C)")
print(f"dtype         : {trials.dtype}")
print(f"value range   : [{trials.min():.3f}, {trials.max():.3f}]")
print(f"mean / std    : {trials.mean():.4f} / {trials.std():.4f}")
print(f"unique labels : {np.unique(labels).tolist()}")
print(f"unique sessions: {np.unique(session_ids).tolist()}")

## 3  Class distribution

In [ ]:
label_series = pd.Series(labels, name="label")
counts = label_series.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(counts.index, counts.values, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Character class")
ax.set_ylabel("Trial count")
ax.set_title(f"Class distribution  (N={N}, {len(counts)} classes)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
print(counts.to_string())

## 4  Session & block structure

In [ ]:
meta_df = pd.DataFrame({
    "trial_id":   trial_ids,
    "session":    session_ids,
    "block":      block_ids,
    "label":      labels,
})

sess_summary = meta_df.groupby("session").agg(
    n_trials=("trial_id", "count"),
    n_blocks=("block",    "nunique"),
    n_classes=("label",   "nunique"),
).reset_index()
print(sess_summary.to_string(index=False))

## 5  Train / val / test splits

In [ ]:
splits = load_splits(cfg)

for split_name, sp in splits.items():
    tr = len(sp["train_idx"])
    va = len(sp["val_idx"])
    te = len(sp["test_idx"])
    total = tr + va + te
    print(f"[{split_name:20s}]  train={tr:5d} ({100*tr/total:.0f}%)  "
          f"val={va:4d} ({100*va/total:.0f}%)  "
          f"test={te:4d} ({100*te/total:.0f}%)  "
          f"total={total}")

## 6  Example trials — mean neural activity per character

For a handful of characters, plot the **population-mean firing rate** (averaged
across trials) as a heatmap over time × channel.

In [ ]:
highlight = ["a", "b", "e", "t", "m", "o", "s", "r"]
t_axis = np.arange(T) * cfg["dt_ms"]  # ms from movement onset

fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharey=True)
axes = axes.ravel()

vmax = 2.5  # colour scale in z-score units
for ax, char in zip(axes, highlight):
    mask = labels == char
    mean_tc = trials[mask].mean(axis=0)  # (T, C)
    im = ax.imshow(
        mean_tc.T,          # (C, T) → channels on y-axis
        aspect="auto",
        origin="lower",
        extent=[t_axis[0], t_axis[-1], 0, C],
        vmin=-vmax, vmax=vmax,
        cmap="RdBu_r",
    )
    ax.set_title(f"'{char}'  (n={mask.sum()})")
    ax.set_xlabel("Time from go-cue (ms)")
    if ax is axes[0] or ax is axes[4]:
        ax.set_ylabel("Channel")

fig.colorbar(im, ax=axes, shrink=0.6, label="z-score")
fig.suptitle("Mean neural activity per character  (channels × time)", y=1.01)
plt.tight_layout()
plt.show()

## 7  Single-trial vs mean: example channel traces

In [ ]:
CHARS_TO_SHOW = ["a", "t", "o"]
TOP_K_CHANNELS = 5    # most discriminative by variance

# Find channels with highest across-class variance in the mean response
class_means = np.stack([trials[labels == c].mean(axis=0) for c in np.unique(labels)], axis=0)  # (31, T, C)
channel_var = class_means.var(axis=0).mean(axis=0)  # (C,)  — time-averaged cross-class variance
top_ch = np.argsort(channel_var)[::-1][:TOP_K_CHANNELS]

fig, axes = plt.subplots(len(CHARS_TO_SHOW), TOP_K_CHANNELS,
                         figsize=(14, 3 * len(CHARS_TO_SHOW)), sharey="row")
for row, char in enumerate(CHARS_TO_SHOW):
    mask = labels == char
    char_trials = trials[mask]   # (n_trials, T, C)
    for col, ch in enumerate(top_ch):
        ax = axes[row, col]
        for tr in char_trials:
            ax.plot(t_axis, tr[:, ch], color="steelblue", alpha=0.15, lw=0.7)
        ax.plot(t_axis, char_trials[:, :, ch].mean(axis=0),
                color="darkred", lw=1.8, label="mean")
        ax.set_title(f"ch {ch}", fontsize=8)
        if col == 0:
            ax.set_ylabel(f"'{char}'  z-score")
        if row == len(CHARS_TO_SHOW) - 1:
            ax.set_xlabel("ms")

fig.suptitle(f"Top-{TOP_K_CHANNELS} most discriminative channels — individual trials (blue) + mean (red)",
             y=1.01)
plt.tight_layout()
plt.show()

## 8  Feature representations

In [ ]:
from utils_features import build_flat, build_temporal, temporal_window_boundaries

all_idx    = np.arange(N)
X_flat     = build_flat(trials, all_idx)       # (N, T*C)
X_temporal = build_temporal(trials, all_idx)   # (N, 3*C)

print(f"Flat feature shape    : {X_flat.shape}   "
      f"({T} time bins × {C} channels)")
print(f"Temporal feature shape: {X_temporal.shape}  "
      f"(3 windows × {C} channels)")

windows = temporal_window_boundaries(cfg)
for name, (s, e) in windows.items():
    print(f"  window '{name}': bins [{s}, {e})  → {(e-s)*cfg['dt_ms']:.0f} ms")

## 9  PCA of flat features — 2-D scatter

In [ ]:
# Use block-aware train split to fit PCA, project all trials
sp = splits["block_aware"]
tr_idx = np.array(sp["train_idx"])

pca = PCA(n_components=10, random_state=42)
pca.fit(X_flat[tr_idx])
Z = pca.transform(X_flat)  # (N, 10)

print(f"Explained variance (top 10 PCs): "
      f"{100*pca.explained_variance_ratio_.cumsum()[-1]:.1f}%")
print("  " + "  ".join(
    f"PC{i+1}:{100*v:.1f}%"
    for i, v in enumerate(pca.explained_variance_ratio_)
))

In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels)
classes = list(le.classes_)
n_classes = len(classes)

cmap = plt.cm.get_cmap("tab20", n_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (pc_x, pc_y, title) in zip(
    axes,
    [(0, 1, "PC1 vs PC2"), (2, 3, "PC3 vs PC4")],
):
    for ci, char in enumerate(classes):
        mask = labels == char
        ax.scatter(Z[mask, pc_x], Z[mask, pc_y],
                   color=cmap(ci), s=8, alpha=0.5, label=char)
    ax.set_xlabel(f"PC{pc_x+1} ({100*pca.explained_variance_ratio_[pc_x]:.1f}%)")
    ax.set_ylabel(f"PC{pc_y+1} ({100*pca.explained_variance_ratio_[pc_y]:.1f}%)")
    ax.set_title(title)

handles = [plt.Line2D([0],[0], marker='o', color='w',
                      markerfacecolor=cmap(i), markersize=6)
           for i in range(n_classes)]
axes[1].legend(handles, classes, fontsize=6, ncol=4,
               bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
fig.suptitle("PCA of flat features (all trials, colour = character class)")
plt.tight_layout()
plt.show()

## 10  Load pre-computed feature NPZs and inspect splits

In [ ]:
processed_dir = Path(cfg["processed_dir"])

for feat_type in ["flat", "temporal"]:
    for split_name in ["block_aware", "random_trial"]:
        npz_path = processed_dir / f"features_{feat_type}_{split_name}.npz"
        if not npz_path.exists():
            print(f"  [missing]  {npz_path.name}")
            continue
        d = np.load(npz_path, allow_pickle=True)
        print(
            f"  {npz_path.name:45s}  "
            f"train={d['X_train'].shape}  "
            f"val={d['X_val'].shape}  "
            f"test={d['X_test'].shape}"
        )

## 11  Saved results overview

In [ ]:
tables_dir = Path(cfg["tables_dir"])

summary_path = tables_dir / "all_models_summary.csv"
if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print(f"all_models_summary.csv — {len(summary_df)} rows, cols: {list(summary_df.columns)}")
    display(summary_df.sort_values("test_acc", ascending=False).head(15)
            if "test_acc" in summary_df.columns
            else summary_df.head(15))
else:
    print(f"[not found] {summary_path}")

for tbl in ["linear_metrics.csv", "nonlinear_metrics.csv", "pca_metrics.csv"]:
    p = tables_dir / tbl
    if p.exists():
        df = pd.read_csv(p)
        print(f"\n{tbl}:")
        display(df.head(10))

## 12  Preprocessing summary

In [ ]:
summary_path = Path(cfg["metadata_dir"]) / "preprocessing_summary.json"
with open(summary_path) as f:
    preproc = json.load(f)

print(f"Sessions processed : {preproc['n_sessions']}")
print(f"Total trials       : {preproc['n_trials']}")
print(f"Excluded doNothing : {preproc['excluded_doNothing_trials']}")
print(f"T_fixed / channels : {preproc['T_fixed']} bins / {preproc['n_channels']} ch")
print(f"N classes          : {preproc['n_classes']}")
print("\nPreprocessing steps:")
for step in preproc["preprocessing"]["steps"]:
    print(f"  {step}")